# LCA creativity metric

**Hypothesis** (Eran & Avivit, meeting 15.9): while an LLM reads a text word by word, its top-k
next-word guesses stay semantically close in **factual** text (low WordNet LCA height throughout)
and keep converging and diverging in **creative** text (high, jumpy LCA).

**Pipeline**
1. WordNet LCA function + sanity check
2. Input texts: Wikipedia (factual), Harry Potter and a 1922 joke book (creative)
3. Choose the measurement points: nouns and verbs (NLTK), without stop/filler words
4. Run each LLM over each text and score the LCA of its top-k guesses at every measurement point
5. Summary table and plots: per model, and per text with all models plus their average
6. Bonus: SVM on the curve shape (leave-one-text-out)

Run top to bottom. By default the notebook **re-uses the saved model outputs** in `results/` (`RUN_MODELS = False`);
set it to `True` to recompute them (~25 min on an RTX 4060).

## 0. Setup

In [1]:
import json
import re
import urllib.parse
import urllib.request
from functools import lru_cache
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from IPython.display import Image, display
from nltk.corpus import stopwords
from nltk.corpus import wordnet as wn

for res in ["wordnet", "stopwords", "averaged_perceptron_tagger_eng", "punkt_tab"]:
    nltk.download(res, quiet=True)

ROOT = Path.cwd()                 # the LCA_Creativity folder (where this notebook lives)
TEXTS = ROOT / "texts"
assert TEXTS.is_dir(), f"open the notebook from the LCA_Creativity folder (cwd = {ROOT})"

# --- experiment settings ------------------------------------------------------------
MODELS = ["gpt2-medium", "EleutherAI/pythia-410m", "Qwen/Qwen2.5-1.5B"]
K = 5                    # top-k guesses per measurement point
USE_ADJ = False          # also measure adjectives
COMPLETE_WORDS = True    # only whole-word guesses (False = old behaviour that let word pieces in, results/k5)
SMOOTH = 10              # rolling-mean window in the plots

# --- what to (re)compute ------------------------------------------------------------
REFETCH_TEXTS = False    # re-download Wikipedia + jokes (Wikipedia changes over time -> different texts)
REEXTRACT_HP = False     # re-cut the Harry Potter excerpts from harrypotter.pdf
RUN_MODELS = True       # re-run the LLMs (slow); False = use the CSVs already in OUT_DIR

OUT_DIR = ROOT / "results" / (f"k{K}" + ("_adj" if USE_ADJ else "") + ("_clean" if COMPLETE_WORDS else ""))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("results folder:", OUT_DIR)

ModuleNotFoundError: No module named 'pandas'

## 1. The LCA function over WordNet

For a set of words $W$, the score is how many is-a (hypernym) levels the **farthest** word has to climb
until all the words meet, "the max distance between the LCA and the lowest word":

$$\text{height}(W) = \min_{A \,\in\, \text{common ancestors}} \; \max_{w \in W} \text{dist}(w, A)$$

`dist` is the shortest hypernym path from any of the word's **2 most common senses**. 2 senses are needed because sense 1 of
*canine* is the tooth, but this can also let a slang sense win (e.g. dog/cat/pet → *person*).
Verbs have many roots in WordNet, so a virtual root joins them.

In [3]:
POS = {"n": wn.NOUN, "v": wn.VERB, "a": wn.ADJ}
VIRTUAL_ROOT = "<ROOT>"  # verbs/adjectives have many roots; join them here


def senses(word, pos, max_senses=2):
    """The word's first `max_senses` senses (WordNet orders them by frequency)."""
    return wn.synsets(word, pos=pos)[:max_senses]


@lru_cache(maxsize=200_000)
def _up_distances(synset):
    """Shortest is-a distance from `synset` to each of its ancestors (incl. itself)."""
    dist = {synset: 0}
    frontier = [synset]
    while frontier:
        nxt = []
        for s in frontier:
            for h in s.hypernyms() + s.instance_hypernyms():
                if h not in dist:
                    dist[h] = dist[s] + 1
                    nxt.append(h)
        frontier = nxt
    roots = [s for s in dist if not (s.hypernyms() or s.instance_hypernyms())]
    dist[VIRTUAL_ROOT] = min(dist[r] for r in roots) + 1
    return dist


def _word_distances(word, pos, max_senses):
    """ancestor -> min distance over the word's allowed senses."""
    out = {}
    for s in senses(word, pos, max_senses):
        for a, d in _up_distances(s).items():
            if d < out.get(a, 1 << 30):
                out[a] = d
    return out


def lca(words, pos="n", max_senses=2):
    """(height, lca_synset) for a set of words, or (None, None) if a word isn't in WordNet for this POS.
    Minimising over senses inside dist() gives the exact answer in O(#ancestors)."""
    words = list(dict.fromkeys(words))
    dists = [_word_distances(w, POS[pos], max_senses) for w in words]
    if not dists or any(not d for d in dists):
        return None, None
    common = set(dists[0]).intersection(*dists[1:])
    best = min(common, key=lambda a: (max(d[a] for d in dists), a == VIRTUAL_ROOT))
    return max(d[best] for d in dists), best


def name_of(anc):
    return anc.name() if hasattr(anc, "name") else anc

### Sanity check (required before continuing)
The meeting's values (1 / 2 / 8) were estimates of what a proper function would give. What matters is the order,
dog/cat ≤ canine/cat ≪ dog/chair, and the function keeps it. dog/cat can't be 1 in WordNet 3.0, because
`cat.n.01` is not under `domestic_animal`, so dog and cat meet at *carnivore*, not at *pet*.

In [4]:
SANITY = [("dog", "cat", 1), ("canine", "cat", 2), ("dog", "chair", 8)]
print(f"{'pair':<16}{'estimate':>9}{'got':>5}  LCA")
for a, b, est in SANITY:
    h, anc = lca([a, b])
    print(f"{a + '/' + b:<16}{est:>9}{h:>5}  {name_of(anc)}")

print()
for words, pos in [(["dog", "cat"], "n"), (["dog", "crocodile"], "n"), (["dog", "cat", "idea"], "n"),
                   (["dog", "cat", "pet"], "n"), (["run", "walk", "jump"], "v")]:
    h, anc = lca(words, pos)
    print(f"{' '.join(words):24s} -> LCA {name_of(anc):18s} height {h}")

pair             estimate  got  LCA
dog/cat                 1    2  carnivore.n.01
canine/cat              2    2  carnivore.n.01
dog/chair               8    6  whole.n.02

dog cat                  -> LCA carnivore.n.01     height 2
dog crocodile            -> LCA vertebrate.n.01    height 5
dog cat idea             -> LCA entity.n.01        height 7
dog cat pet              -> LCA organism.n.01      height 4
run walk jump            -> LCA <ROOT>             height 2


## 2. Input texts
| file prefix | source |
|---|---|
| `factual_wiki_*` | ~700-word sections from 8 Wikipedia articles (MediaWiki API) |
| `creative_jokes_*` | 2 excerpts (⅓ and ⅔ into the book) of *Jokes For All Occasions* (1922, Gutenberg #21084), unfiltered |
| `creative_hp1..7` | ~700 words from the middle of each Harry Potter book, cut from the local `harrypotter.pdf` (copyrighted, local only) |
| `creative_harry_potter` | the opening of book 1, added by hand |

The texts are already in `texts/`. The cells below only re-create them if the flags in the setup cell are on.

In [ ]:
N_WORDS = 700
UA = {"User-Agent": "lca-creativity-student-project/0.1"}
WIKI = {"factual_wiki_wordnet": "WordNet",
        "factual_wiki_photosynthesis": "Photosynthesis",
        "factual_wiki_volcano": "Volcano",
        "factual_wiki_mitochondrion": "Mitochondrion",
        "factual_wiki_plate_tectonics": "Plate tectonics",
        "factual_wiki_black_hole": "Black hole",
        "factual_wiki_vaccine": "Vaccine",
        "factual_wiki_copper": "Copper"}


def get(url):
    with urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=60) as r:
        return r.read().decode("utf-8")


def first_words(text, n=N_WORDS):
    cut = " ".join(text.split()[:n])
    return cut[:cut.rfind(".") + 1] or cut  # end on a sentence boundary


def wiki(title):
    q = urllib.parse.urlencode(dict(action="query", prop="extracts", explaintext=1, format="json",
                                    titles=title, redirects=1))
    page = next(iter(json.loads(get(f"https://en.wikipedia.org/w/api.php?{q}"))["query"]["pages"].values()))
    body = re.split(r"\n==\s*(See also|References|Notes)\s*==", page["extract"])[0]
    body = re.sub(r"\n=+[^=\n]+=+\n", "\n", body)  # drop section headings
    return first_words(" ".join(body.split()))


def jokes(fractions=(1 / 3, 2 / 3)):
    raw = get("https://www.gutenberg.org/cache/epub/21084/pg21084.txt")
    body = raw.split("*** START OF")[1].split("*** END OF")[0]
    paras = [" ".join(p.split()) for p in body.split("\r\n\r\n") if p.strip()]
    paras = [p for p in paras if len(p.split()) > 3 and not p.isupper()]  # drop headings only
    return [first_words("\n\n".join(paras[int(len(paras) * f):][:40])) for f in fractions]


if REFETCH_TEXTS:
    for name, title in WIKI.items():
        (TEXTS / f"{name}.txt").write_text(wiki(title), encoding="utf-8")
    for name, text in zip(["creative_jokes_for_all_occasions", "creative_jokes_2"], jokes()):
        (TEXTS / f"{name}.txt").write_text(text, encoding="utf-8")

In [ ]:
BOOK_STARTS = [11, 281, 572, 948, 1569, 2414, 2975]  # 0-based PDF pages of each book's "CHAPTER ONE"


def clean_pdf_text(t):
    """The PDF text layer uses tabs for spaces, U+FFFD for curly quotes/apostrophes and hyphenates lines."""
    t = t.replace("\t", " ")
    t = re.sub(r"(?<=[A-Za-z])\ufffd(?=[A-Za-z])", "'", t)  # I\ufffdm -> I'm
    t = t.replace("\ufffd", '"')
    t = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", t)  # de-hyphenate line breaks
    t = re.sub(r"\n\s*\d+\s*\n", "\n", t)  # bare page numbers
    return re.sub(r"\s+", " ", t).strip()


def hp_excerpt(reader, start, end):
    mid = (start + end) // 2
    text = clean_pdf_text(" ".join(reader.pages[p].extract_text() or "" for p in range(mid, mid + 6)))
    first = re.search(r"(?<=[.!?\"]) (?=[A-Z\"])", text)  # begin at a sentence start
    cut = " ".join(text[first.end() if first else 0:].split()[:N_WORDS])
    return cut[:max(cut.rfind(". "), cut.rfind('." ')) + 1] or cut


if REEXTRACT_HP:
    from pypdf import PdfReader
    reader = PdfReader(ROOT / "harrypotter.pdf")
    bounds = BOOK_STARTS + [len(reader.pages)]
    for book, (s, e) in enumerate(zip(bounds, bounds[1:]), 1):
        (TEXTS / f"creative_hp{book}.txt").write_text(hp_excerpt(reader, s, e), encoding="utf-8")


def kind(text_name):
    return "factual" if text_name.startswith("factual") else "creative"


texts = {p.stem: p.read_text(encoding="utf-8") for p in sorted(TEXTS.glob("*.txt"))}
pd.DataFrame([(n, kind(n), len(t.split())) for n, t in texts.items()], columns=["text", "kind", "words"])

## 3. Where to measure: nouns and verbs without fillers
NLTK tags every word's part of speech. A word becomes a measurement point only if:
- it's a **noun or verb**, not a name (adjectives only with `USE_ADJ`);
- it isn't a stop or filler word (*the, is, also, being*…) or an auxiliary verb (*be / have / do*);
- it exists in WordNet as that part of speech.

The model still reads the **whole** text; these rules only decide where to measure and which guesses count.

Guesses get two extra filters:
- **base form**: *hands* → *hand*, not the 'hands' = guardianship synset; *saw* → *see*;
- **main usage**: the target part of speech must be the guess's most frequent usage in WordNet, so *back*, *short* and *full* don't count as nouns.

In [5]:
FILLERS = {"also", "being", "however", "thus", "therefore", "would", "could", "should",
           "may", "might", "must", "shall", "said", "say", "says", "get", "got", "make",
           "made", "go", "went", "one", "two", "like", "well", "even", "much", "many", "us"}
STOP = set(stopwords.words("english")) | FILLERS
AUX_LEMMAS = {"be", "have", "do"}


def base_lemma(word, pos):
    """WordNet lemma, preferring the base form ('hands' -> 'hand', 'saw' -> 'see')."""
    forms = wn._morphy(word, pos)
    if not forms:
        return None
    inflected = [f for f in forms if f != word]
    return min(inflected or forms, key=len)


def _sense_count(word, pos):
    lemma = base_lemma(word, pos)
    if lemma is None:
        return -1
    return sum(l.count() for s in wn.synsets(lemma, pos) for l in s.lemmas() if l.name().lower() == lemma)


@lru_cache(maxsize=None)
def candidate_lemma(word, pos):
    """Lemma of a guess if `pos` is its dominant usage in WordNet (by sense frequency), else None."""
    lemma = base_lemma(word, POS[pos])
    if lemma is None or (pos == "v" and lemma in AUX_LEMMAS):
        return None
    mine = _sense_count(word, POS[pos])
    others = max(_sense_count(word, p) for p in (wn.NOUN, wn.VERB, wn.ADJ, wn.ADV) if p != POS[pos])
    return lemma if mine >= others else None


def tag_to_pos(tag, use_adj):
    if tag.startswith("NN") and not tag.startswith("NNP"):
        return "n"
    if tag.startswith("VB"):
        return "v"
    if use_adj and tag.startswith("JJ"):
        return "a"
    return None


def content_words(text, use_adj=False):
    """[(char_start, word, pos)] of the measurement points."""
    out, cursor = [], 0
    for sent in nltk.sent_tokenize(text):
        sent_start = text.find(sent, cursor)
        cursor = sent_start + len(sent)
        toks = [(m.start() + sent_start, m.group()) for m in re.finditer(r"[A-Za-z]+", sent)]
        for (start, w), (_, tag) in zip(toks, nltk.pos_tag([w for _, w in toks])):
            pos, lw = tag_to_pos(tag, use_adj), w.lower()
            if pos is None or lw in STOP:
                continue
            lemma = base_lemma(lw, POS[pos])
            if lemma is None or (pos == "v" and lemma in AUX_LEMMAS):
                continue
            out.append((start, w, pos))
    return out


content_words("The physician replied very coldly that the fees must be paid in money, not in gratitude.")

[(43, 'fees', 'n'),
 (56, 'paid', 'v'),
 (64, 'money', 'n'),
 (78, 'gratitude', 'n')]

## 4. Measuring a text with an LLM

**"Feeding word by word" = teacher forcing.** One causal forward pass over the text gives, at the token just
before each word, exactly the distribution the model would produce if it were fed only the text up to there.
Long texts use a sliding window that always keeps at least half a window of context.

At each measurement point:
1. Walk down the model's top 300 next-token predictions. Keep tokens that start a new alphabetic word and pass `candidate_lemma`
   for the **same part of speech** as the real word.
2. **Whole-word check.** Tokens are often only the first piece of a longer word
   (*photosynthesis* = `_photos` + `ynthesis`), and pieces like *bi*, *e*, *phot* are WordNet nouns (bismuth, vitamin E…).
   A guess is kept only if, after it, the model puts ≥ 50% probability on a word boundary (space, punctuation, newline, end-of-text).
   This is one extra batched pass per point, and it's the expensive step.
3. Take the first `K` different lemmas (*word*/*words* merge and their probabilities add up). Skip the point if fewer than 2 remain.
4. Score their LCA height. The real word is **not** part of the score; its probability is only recorded (`actual_word_prob`).

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


class LCAMeter:
    def __init__(self, model_name, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.tok = AutoTokenizer.from_pretrained(model_name)
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(self.device).eval()
        self.window = min(getattr(self.model.config, "max_position_embeddings", 1024) or 1024, 2048)
        self._build_vocab_tables()

    def _build_vocab_tables(self):
        """vocab_word: token id -> lowercase word, for tokens that start a new alphabetic word.
        boundary: tokens that END the current word when they come next (new word, punctuation, newline, EOS)."""
        self.vocab_word = {}
        self.boundary = torch.zeros(self.model.get_output_embeddings().weight.shape[0], dtype=torch.bool)
        for tid, piece in enumerate(self.tok.convert_ids_to_tokens(list(range(len(self.tok))))):
            if not isinstance(piece, str):
                continue
            s = self.tok.convert_tokens_to_string([piece])
            self.boundary[tid] = piece[:1] in ("Ġ", "▁", "Ċ") or (s[:1] != "" and not s[:1].isalnum())
            w = s.strip()
            if piece[:1] in ("Ġ", "▁") and w.isalpha() and w.lower() not in STOP:
                self.vocab_word[tid] = w.lower()
        if self.tok.eos_token_id is not None:
            self.boundary[self.tok.eos_token_id] = True
        self.boundary = self.boundary.to(self.device)

    @torch.no_grad()
    def _completes_word(self, seq, j, tids, ctx=128):
        """Per candidate token at position j: P(next token is a word boundary) >= 0.5 ?"""
        context = seq[max(0, j - ctx):j]
        batch = torch.tensor([context + [t] for t in tids], device=self.device)
        # logits_to_keep=1: only the last position's vocab scores (all positions over Qwen's 152k vocab
        # would be ~600 MB per batch and spill out of an 8 GB GPU)
        probs = torch.softmax(self.model(batch, logits_to_keep=1).logits[:, -1].float(), -1)
        return (probs[:, self.boundary].sum(-1) >= 0.5).tolist()

    @torch.no_grad()
    def _predictions(self, seq, top_n):
        """For every position j>=1: top_n (probs, ids) of P(token_j | seq[:j]) and P(actual token)."""
        n, stride = len(seq), self.window // 2
        top_p = torch.full((n, top_n), float("nan"))
        top_i = torch.zeros((n, top_n), dtype=torch.long)
        actual = torch.full((n,), float("nan"))
        done = 1
        while done < n:
            end = min(self.window if done == 1 else done + stride, n)
            start = max(0, end - self.window)
            logits = self.model(torch.tensor([seq[start:end]], device=self.device)).logits[0]
            probs = torch.softmax(logits[done - 1 - start:end - 1 - start].float(), -1)
            p, i = probs.topk(top_n, -1)
            top_p[done:end], top_i[done:end] = p.cpu(), i.cpu()
            tgt = torch.tensor(seq[done:end], device=self.device)
            actual[done:end] = probs.gather(1, tgt[:, None])[:, 0].cpu()
            done = end
        return top_p, top_i, actual

    def _candidates(self, seq, j, top_p, top_i, pos, k, complete_words):
        """Top-k guesses at position j as {lemma: summed probability}, in rank order."""
        pool = []  # (tid, lemma, p); collect up to 3k lemmas in case the whole-word check drops some
        for p, tid in zip(top_p[j].tolist(), top_i[j].tolist()):
            w = self.vocab_word.get(tid)
            lemma = candidate_lemma(w, pos) if w else None
            if lemma:
                pool.append((tid, lemma, p))
                if len({l for _, l, _ in pool}) >= 3 * k:
                    break
        if complete_words and pool:
            ok = self._completes_word(seq, j, [tid for tid, _, _ in pool])
            pool = [c for c, keep in zip(pool, ok) if keep]
        cands = {}
        for _, lemma, p in pool:
            if lemma in cands:
                cands[lemma] += p  # 'word' and 'words' are the same guess
            elif len(cands) < k:
                cands[lemma] = p
        return cands

    def run(self, text, k=5, top_n=300, use_adj=False, max_senses=2, complete_words=True):
        """One row per measurement point."""
        enc = self.tok(text, return_offsets_mapping=True, add_special_tokens=False)
        ids, offsets = enc["input_ids"], enc["offset_mapping"]
        prefix = [self.tok.bos_token_id] if self.tok.bos_token_id is not None else []
        seq = prefix + ids
        top_p, top_i, actual = self._predictions(seq, top_n)
        ends = np.array([e for _, e in offsets])

        rows = []
        for idx, (cstart, word, pos) in enumerate(content_words(text, use_adj)):
            t = int(np.searchsorted(ends, cstart, side="right"))  # token containing the word start
            j = t + len(prefix)
            if t >= len(ids) or j == 0 or text[offsets[t][0]:cstart].strip():
                continue  # no context yet, or word not at a token boundary (e.g. "(dog")
            cands = self._candidates(seq, j, top_p, top_i, pos, k, complete_words)
            if len(cands) < 2:
                continue
            h, anc = lca(list(cands), pos, max_senses)
            rows.append(dict(word_idx=idx, char=cstart, word=word, pos=pos, lca_height=h, lca=name_of(anc),
                             candidates=" ".join(cands), cand_mass=sum(cands.values()),
                             actual_word_prob=actual[j].item()))
        return pd.DataFrame(rows)

    def free(self):
        del self.model
        torch.cuda.empty_cache()

### Run every model over every text
Writes `OUT_DIR/<model>__<text>.csv`. This only runs with `RUN_MODELS = True` (~25 min; Qwen ≈ 1 min per text).

In [ ]:
def slug(model):
    return model.replace("/", "_")


if RUN_MODELS:
    for model in MODELS:
        meter = LCAMeter(model)
        for name, text in texts.items():
            df = meter.run(text, k=K, use_adj=USE_ADJ, complete_words=COMPLETE_WORDS)
            df.to_csv(OUT_DIR / f"{slug(model)}__{name}.csv", index=False)
            print(f"{model:28s} {name:40s} n={len(df)}  mean={df['lca_height'].mean():.2f}", flush=True)
        meter.free()

data = {m: {f.stem.split("__")[1]: pd.read_csv(f) for f in sorted(OUT_DIR.glob(f"{slug(m)}__*.csv"))}
        for m in MODELS}  # {model: {text: df}}
print({m: len(d) for m, d in data.items()}, "texts per model")
data[MODELS[0]]["factual_wiki_wordnet"].head(8)

## 5. Summary and plots
- `summary.csv`: per model × text. `mean`, `var`, `std`, `mean_abs_jump` (mean |change| between consecutive points),
  `p90`, the noun-only mean/var, the number of points `n`, and `verb_frac`.
- `<model>.png`: factual texts on the left, creative on the right.
- `per_text/<text>.png`: all models plus their average, aligned word by word.

In [ ]:
LABEL = "LCA height (max)"
COLORS = {"factual": "#2a6fdb", "creative": "#e0662f"}
MODEL_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"]  # categorical slots 1-3, distinguishable for colour-blind readers
AVG_COLOR = "#1f1f1f"


def summarize(df):
    h = df["lca_height"]
    return dict(n=len(h), mean=h.mean(), var=h.var(), std=h.std(),
                mean_abs_jump=h.diff().abs().mean(), p90=h.quantile(0.9))


def roll(s):
    return s.rolling(SMOOTH, center=True, min_periods=1).mean()


rows = []
for m, per_text in data.items():
    for name, df in per_text.items():
        s, nouns = summarize(df), summarize(df[df["pos"] == "n"])
        n = s.pop("n")
        rows.append(dict(model=m, text=name, kind=kind(name), **s, noun_mean=nouns["mean"],
                         noun_var=nouns["var"], n=n, verb_frac=(df["pos"] == "v").mean()))
summary = pd.DataFrame(rows)
summary.to_csv(OUT_DIR / "summary.csv", index=False)
print("verb fraction:", summary.groupby("kind")["verb_frac"].mean().round(2).to_dict())
summary.groupby(["model", "kind"])[["mean", "var", "mean_abs_jump", "noun_mean", "noun_var"]].mean().round(2)

In [ ]:
def plot_model(model, results, out):
    """Factual texts in the left column, creative texts in the right one."""
    cols = [[n for n in results if kind(n) == k] for k in ("factual", "creative")]
    n_rows = max(len(c) for c in cols)
    fig, axes = plt.subplots(n_rows, 2, figsize=(20, 2.1 * n_rows), sharey=True, squeeze=False)
    for ax in axes.flat:
        ax.set_visible(False)
    for c, names in enumerate(cols):
        for r, name in enumerate(names):
            ax, df = axes[r, c], results[name]
            color = COLORS[kind(name)]
            ax.set_visible(True)
            ax.plot(df["lca_height"].values, color=color, alpha=0.35, lw=0.9)
            ax.plot(roll(df["lca_height"]).values, color=color, lw=2)
            s = summarize(df)
            ax.set_title(f"{name}   mean={s['mean']:.2f}  var={s['var']:.2f}  |jump|={s['mean_abs_jump']:.2f}",
                         fontsize=9, loc="left")
            ax.set_ylabel(LABEL)
            ax.grid(alpha=0.3)
        if names:
            axes[len(names) - 1, c].set_xlabel("content word # along the text")
    fig.suptitle(f"{model} — {LABEL} of top-k completions (thin: raw, thick: rolling mean {SMOOTH})")
    fig.tight_layout(rect=(0, 0, 1, 0.985))
    fig.savefig(out, dpi=110)
    plt.close(fig)


def plot_text(name, out):
    """Every model's curve for one text + the cross-model average, aligned on word_idx."""
    table = pd.DataFrame({m: d[name].set_index("word_idx")["lca_height"] for m, d in data.items() if name in d})
    table = table.sort_index()  # rows: content word, cols: model
    fig, ax = plt.subplots(figsize=(14, 4.2))
    for (m, s), color in zip(table.items(), MODEL_COLORS):
        ax.plot(table.index, s, color=color, lw=0.7, alpha=0.18)
        ax.plot(table.index, roll(s), color=color, lw=1.6, alpha=0.9, label=f"{m}  (mean {s.mean():.2f})")
    avg = table.mean(axis=1)
    ax.plot(table.index, roll(avg), color=AVG_COLOR, lw=3,
            label=f"average of models  (mean {avg.mean():.2f}, var {avg.var():.2f})")
    ax.set_title(f"{name}  [{kind(name)}]  — {LABEL} along the text (thin: raw, thick: rolling mean {SMOOTH})",
                 loc="left", fontsize=11)
    ax.set_xlabel("content word # along the text")
    ax.set_ylabel(LABEL)
    ax.set_ylim(0, 16)
    ax.grid(alpha=0.25, lw=0.6)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(loc="upper left", fontsize=8.5, frameon=False, ncol=2)
    fig.tight_layout()
    fig.savefig(out, dpi=110)
    plt.close(fig)


for m in MODELS:
    plot_model(m, data[m], OUT_DIR / f"{slug(m)}.png")
(OUT_DIR / "per_text").mkdir(exist_ok=True)
for name in sorted({t for d in data.values() for t in d}):
    plot_text(name, OUT_DIR / "per_text" / f"{name}.png")

display(Image(OUT_DIR / f"{slug(MODELS[-1])}.png", width=1100))

In [ ]:
for name in ["factual_wiki_vaccine", "creative_hp6"]:
    display(Image(OUT_DIR / "per_text" / f"{name}.png", width=1100))

## 6. Bonus: can an SVM tell factual from creative by the curve's shape?
Each curve is cut into windows of 40 points (step 20). Each window gets 6 features: level, spread, mean |jump|, max |jump|,
share ≥ 8, and the rate of direction changes (convergence/divergence). Evaluation is **leave-one-text-out**, so the
windows of a text are never in both train and test. `pos="n"` uses nouns only, which removes the verb confound
(verbs sit ~3 levels lower in WordNet, and creative texts have ~2× as many verbs).

In [ ]:
from sklearn.metrics import classification_report
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

WIN, STEP = 40, 20


def features(h):
    d = np.diff(h)
    return [h.mean(), h.var(), np.abs(d).mean(), np.abs(d).max(), (h >= 8).mean(),
            np.mean(np.sign(d[1:]) != np.sign(d[:-1]))]  # direction changes = convergence/divergence


def classify(pos=None):
    X, y, groups = [], [], []
    for f in sorted(OUT_DIR.glob("*__*.csv")):
        text = f.stem.split("__")[1]
        df = pd.read_csv(f)
        h = df.loc[df["pos"] == pos if pos else slice(None), "lca_height"].to_numpy(float)
        for s in range(0, len(h) - WIN + 1, STEP):
            X.append(features(h[s:s + WIN]))
            y.append(kind(text))
            groups.append(text)
    X, y = np.array(X), np.array(y)
    texts_per_class = pd.Series(dict(zip(groups, y))).value_counts()
    if len(texts_per_class) < 2 or texts_per_class.min() < 2:
        raise ValueError(f"need >= 2 texts of each kind for leave-one-text-out, got {texts_per_class.to_dict()}")
    print(f"{len(X)} windows from {len(set(groups))} texts: {dict(zip(*np.unique(y, return_counts=True)))}")
    clf = make_pipeline(StandardScaler(), SVC(kernel="rbf", class_weight="balanced"))
    pred = cross_val_predict(clf, X, y, groups=groups, cv=LeaveOneGroupOut())
    print(classification_report(y, pred, zero_division=0))


print("=== nouns + verbs"); classify()
print("=== nouns only");    classify(pos="n")

## 7. Findings so far (`results/k5_clean`)
- **Factual scores higher than creative** in all 3 models (≈5.7 vs ≈4.8): the opposite of the hypothesis.
- Most of the gap is the **verb confound**: creative ≈ 47% verbs vs factual ≈ 26%, and verbs score ~3 levels lower.
  With **nouns only** the gap nearly disappears (≈6.5 vs ≈6.4), and the SVM drops from ~86% to chance (~53%).
- The models *do* narrow down more on factual text (P(real word) ≈ 0.43 vs 0.27), but WordNet scatters one
  scientific topic across distant branches (*crust, continent, mantle, surface*), so focused guesses still get a high LCA.
- The 3 models follow almost the same curve on every text: the curve's shape comes from the text, not the model.
- Things to discuss: distances from embeddings instead of WordNet depth; the "general word" quirk (*animal* among
  *dog, wolf*); slang senses winning (dog/cat/pet → *person*); analysing nouns only.

Note: when two ancestors tie, which one is shown in the `lca` **name** column depends on Python's per-run hash order.
`lca_height` is never affected.